# AI-Powered Business Intelligence

A local-first business intelligence notebook using **Llama 3.2 3B Instruct (GGUF, Q4_K_M)**, `llama.cpp`, LangChain, SQLite and pandas.

It keeps the business database, SQL and calculation tools, agent, charts, PDF report, and Colab export workflow. The language model runs locally in the Colab runtime; no hosted model API key is required.

**Runtime:** Google Colab CPU. The quantised model file is approximately 2.02 GB; model loading and generation may take time on CPU.

## 1. Install dependencies

## Cell 1

In [ ]:
%pip install -q --upgrade "langchain>=1.0,<2.0" langchain-community huggingface_hub pandas matplotlib reportlab nbformat

## Cell 2

In [ ]:
%pip install -q --prefer-binary llama-cpp-python --extra-index-url https://abetlen.github.io/llama-cpp-python/whl/cpu

## 2. Initialise the project

## Cell 3

In [ ]:
import os
import json
import sqlite3
import importlib.metadata as metadata
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt

PROJECT_DIR = Path("/content/ai_business_analyst")
DATA_DIR = PROJECT_DIR / "data"
OUTPUT_DIR = PROJECT_DIR / "outputs"

DATA_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

DB_PATH = DATA_DIR / "business.db"

print("Project:", PROJECT_DIR)
print("Database:", DB_PATH)
print("Outputs:", OUTPUT_DIR)

## Cell 4

In [ ]:
packages = [
    "langchain",
    "langchain-community",
    "llama-cpp-python",
    "huggingface_hub",
    "pandas",
    "matplotlib",
    "reportlab",
]

for package in packages:
    try:
        print(f"{package}: {metadata.version(package)}")
    except metadata.PackageNotFoundError:
        print(f"{package}: NOT INSTALLED")

## 3. Download and load Llama 3.2 3B

## Cell 5

In [ ]:
from huggingface_hub import hf_hub_download

MODEL_REPO = "bartowski/Llama-3.2-3B-Instruct-GGUF"
MODEL_FILE = "Llama-3.2-3B-Instruct-Q4_K_M.gguf"

model_path = hf_hub_download(
    repo_id=MODEL_REPO,
    filename=MODEL_FILE,
)

print("Model downloaded to:", model_path)

## Cell 6

In [ ]:
import os
from langchain_community.chat_models import ChatLlamaCpp

llm = ChatLlamaCpp(
    model_path=model_path,
    temperature=0.0,
    max_tokens=512,
    n_ctx=4096,
    n_threads=max(2, os.cpu_count() or 2),
    verbose=False,
)

print("Llama 3.2 3B loaded successfully.")

## Cell 7

In [ ]:
response = llm.invoke("What is 125 multiplied by 8? Reply with the answer and one short calculation.")
print(response.content)

## 4. Create the business database

## Cell 8

In [ ]:
data = {
    "product": [
        "Laptop", "Phone", "Tablet", "Monitor", "Keyboard", "Mouse",
        "Laptop", "Phone", "Tablet", "Monitor", "Keyboard", "Mouse"
    ],
    "month": [
        "January", "January", "January", "January", "January", "January",
        "February", "February", "February", "February", "February", "February"
    ],
    "revenue": [
        15000, 9000, 6000, 5000, 2000, 1000,
        18000, 11000, 7000, 5500, 2500, 1200
    ],
    "cost": [
        10000, 6000, 4000, 3000, 1000, 400,
        12000, 7000, 4500, 3200, 1200, 500
    ],
}

df = pd.DataFrame(data)

with sqlite3.connect(DB_PATH) as conn:
    df.to_sql("sales", conn, if_exists="replace", index=False)

print("Sample sales database created.")
display(df.head())

## Cell 9

In [ ]:
with sqlite3.connect(DB_PATH) as conn:
    print(pd.read_sql_query("SELECT * FROM sales LIMIT 5", conn))
    print("\\nRows in sales table:", pd.read_sql_query(
        "SELECT COUNT(*) AS row_count FROM sales", conn
    ).iloc[0, 0])

## 5. Define the business intelligence tools

## Cell 10

In [ ]:
from langchain.tools import tool

@tool
def query_database(sql: str) -> str:
    """Run a read-only SELECT query against the SQLite sales table.

    Table: sales
    Columns: product, month, revenue, cost
    """
    cleaned_sql = sql.strip()

    if not cleaned_sql.lower().startswith("select"):
        return "Error: Only SELECT statements are permitted."

    try:
        with sqlite3.connect(f"file:{DB_PATH}?mode=ro", uri=True) as conn:
            result = pd.read_sql_query(cleaned_sql, conn)
        return result.to_json(orient="records")
    except Exception as exc:
        return f"Database error: {exc}"


@tool
def calculate_profit(revenue: float, cost: float) -> float:
    """Calculate profit by subtracting cost from revenue."""
    return float(revenue) - float(cost)


@tool
def generate_sales_chart() -> str:
    """Create a bar chart of total revenue by product and save it as a PNG."""
    with sqlite3.connect(DB_PATH) as conn:
        chart_data = pd.read_sql_query(
            """
            SELECT product, SUM(revenue) AS revenue
            FROM sales
            GROUP BY product
            ORDER BY revenue DESC
            """,
            conn,
        )

    chart_path = OUTPUT_DIR / "sales_chart.png"

    plt.figure(figsize=(9, 5))
    plt.bar(chart_data["product"], chart_data["revenue"])
    plt.title("Revenue by Product")
    plt.xlabel("Product")
    plt.ylabel("Revenue")
    plt.tight_layout()
    plt.savefig(chart_path, dpi=150)
    plt.close()

    return f"Chart saved to {chart_path}"


tools = [query_database, calculate_profit, generate_sales_chart]
print(f"{len(tools)} tools registered:", [item.name for item in tools])

## 6. Create the AI Business Analyst agent

## Cell 11

In [ ]:
from langchain.agents import create_agent

SYSTEM_PROMPT = """
You are an AI Business Analyst.

The SQLite database contains one table named sales with these columns:
product, month, revenue, cost.

Responsibilities:
1. Retrieve business figures using the SQL tool.
2. Use the profit tool for profit calculations.
3. Generate charts with the chart tool when requested.
4. Explain findings clearly and concisely.

Rules:
- Never invent business figures.
- Use actual tool results for data-dependent answers.
- Only use read-only SQL SELECT queries.
- Do not invent table or column names.
- Verify calculations before reporting them.
"""

agent = create_agent(
    model=llm,
    tools=tools,
    system_prompt=SYSTEM_PROMPT,
)

print("AI Business Analyst agent created.")

## Cell 12

In [ ]:
def ask_business_agent(question: str):
    """Send a question to the business analyst and display its response."""
    result = agent.invoke({
        "messages": [{"role": "user", "content": question}]
    })

    for message in result["messages"]:
        if message.type == "ai" and message.tool_calls:
            print("Tools called:", message.tool_calls)
        elif message.type == "tool":
            print("Tool result:", message.content)
        elif message.type == "ai":
            print("\\nBusiness Analyst:", message.content)

    return result

print("Business Analyst is ready.")

## 7. Test database analysis

## Cell 13

In [ ]:
ask_business_agent(
    "Find the total revenue across all products using the database."
)

## Cell 14

In [ ]:
ask_business_agent(
    "Compare total revenue between January and February. "
    "Use the database and explain which month performed better."
)

## Cell 15

In [ ]:
monthly_revenue = query_database.invoke({
    "sql": """
        SELECT month, SUM(revenue) AS total_revenue
        FROM sales
        GROUP BY month
        ORDER BY CASE month
            WHEN 'January' THEN 1
            WHEN 'February' THEN 2
        END
    """
})

print(monthly_revenue)

## 8. Generate sales visualisations

## Cell 16

In [ ]:
chart_result = generate_sales_chart.invoke({})
print(chart_result)

from IPython.display import display, Image
display(Image(filename=str(OUTPUT_DIR / "sales_chart.png")))

## Cell 17

In [ ]:
ask_business_agent(
    "Generate a bar chart showing total revenue by product using the chart tool."
)
display(Image(filename=str(OUTPUT_DIR / "sales_chart.png")))

## Cell 18

In [ ]:
summary = json.loads(query_database.invoke({
    "sql": """
        SELECT SUM(revenue) AS revenue, SUM(cost) AS cost
        FROM sales
    """
}))[0]

total_profit = calculate_profit.invoke({
    "revenue": summary["revenue"],
    "cost": summary["cost"],
})

print("Total revenue:", summary["revenue"])
print("Total cost:", summary["cost"])
print("Total profit:", total_profit)

## Cell 19

In [ ]:
monthly = json.loads(query_database.invoke({
    "sql": """
        SELECT month, SUM(revenue) AS revenue, SUM(cost) AS cost
        FROM sales
        GROUP BY month
        ORDER BY CASE month WHEN 'January' THEN 1 ELSE 2 END
    """
}))

for row in monthly:
    profit = calculate_profit.invoke({
        "revenue": row["revenue"],
        "cost": row["cost"],
    })
    print(f"\\n{row['month']}")
    print("Revenue:", row["revenue"])
    print("Cost:", row["cost"])
    print("Profit:", profit)

## Cell 20

In [ ]:
products = json.loads(query_database.invoke({
    "sql": """
        SELECT product, SUM(revenue) AS revenue, SUM(cost) AS cost
        FROM sales
        GROUP BY product
        ORDER BY revenue DESC
    """
}))

for row in products:
    profit = calculate_profit.invoke({
        "revenue": row["revenue"],
        "cost": row["cost"],
    })
    margin = (profit / row["revenue"] * 100) if row["revenue"] else 0

    print(
        f"{row['product']} | Revenue: {row['revenue']} "
        f"| Profit: {profit} | Margin: {margin:.2f}%"
    )

## Cell 21

In [ ]:
product_names = [row["product"] for row in products]
product_profits = [
    calculate_profit.invoke({
        "revenue": row["revenue"],
        "cost": row["cost"],
    })
    for row in products
]

plt.figure(figsize=(9, 5))
plt.bar(product_names, product_profits)
plt.title("Total Profit by Product")
plt.xlabel("Product")
plt.ylabel("Profit")
plt.tight_layout()

profit_chart_path = OUTPUT_DIR / "profit_by_product.png"
plt.savefig(profit_chart_path, dpi=150)
plt.show()

print("Saved:", profit_chart_path)

## Cell 22

In [ ]:
top_product = json.loads(query_database.invoke({
    "sql": """
        SELECT product, SUM(revenue) AS revenue
        FROM sales
        GROUP BY product
        ORDER BY revenue DESC
        LIMIT 1
    """
}))[0]

print("Highest-revenue product:", top_product["product"])
print("Revenue:", top_product["revenue"])

## 9. Prepare a business performance summary

## Cell 23

In [ ]:
monthly_summary = json.loads(query_database.invoke({
    "sql": """
        SELECT month, SUM(revenue) AS revenue, SUM(cost) AS cost
        FROM sales
        GROUP BY month
    """
}))

print("BUSINESS PERFORMANCE SUMMARY")
print("----------------------------")
print("Total revenue:", summary["revenue"])
print("Total cost:", summary["cost"])
print("Total profit:", total_profit)
print("Top product:", top_product["product"])
print("Top product revenue:", top_product["revenue"])
print("\\nMONTHLY PERFORMANCE")

for row in monthly_summary:
    month_profit = calculate_profit.invoke({
        "revenue": row["revenue"],
        "cost": row["cost"],
    })
    print(
        f"{row['month']}: Revenue {row['revenue']}, "
        f"Cost {row['cost']}, Profit {month_profit}"
    )

## 10. Generate a PDF report

## Cell 24

In [ ]:
%pip install -q reportlab

## Cell 25

In [ ]:
from reportlab.lib import colors
from reportlab.lib.pagesizes import A4
from reportlab.lib.styles import getSampleStyleSheet
from reportlab.lib.units import cm
from reportlab.platypus import (
    SimpleDocTemplate, Paragraph, Spacer, Table, TableStyle, Image as RLImage
)

pdf_path = OUTPUT_DIR / "business_performance_report.pdf"

doc = SimpleDocTemplate(
    str(pdf_path),
    pagesize=A4,
    rightMargin=2 * cm,
    leftMargin=2 * cm,
    topMargin=2 * cm,
    bottomMargin=2 * cm,
)

styles = getSampleStyleSheet()
story = [
    Paragraph("Business Performance Report", styles["Title"]),
    Spacer(1, 12),
    Paragraph("Financial overview and sales performance", styles["Heading2"]),
    Spacer(1, 16),
]

report_rows = [
    ["Metric", "Result"],
    ["Total Revenue", f"{summary['revenue']:,.0f}"],
    ["Total Cost", f"{summary['cost']:,.0f}"],
    ["Total Profit", f"{total_profit:,.0f}"],
    ["Highest Revenue Product", top_product["product"]],
    ["Highest Product Revenue", f"{top_product['revenue']:,.0f}"],
]

for row in monthly_summary:
    report_rows.append([
        f"{row['month']} Revenue",
        f"{row['revenue']:,.0f}",
    ])

table = Table(report_rows, colWidths=[9 * cm, 7 * cm])
table.setStyle(TableStyle([
    ("BACKGROUND", (0, 0), (-1, 0), colors.HexColor("#243B53")),
    ("TEXTCOLOR", (0, 0), (-1, 0), colors.white),
    ("FONTNAME", (0, 0), (-1, 0), "Helvetica-Bold"),
    ("GRID", (0, 0), (-1, -1), 0.5, colors.grey),
    ("PADDING", (0, 0), (-1, -1), 9),
    ("BACKGROUND", (0, 1), (-1, -1), colors.HexColor("#F1F5F9")),
]))

story.extend([table, Spacer(1, 20)])
story.append(Paragraph("Key Findings", styles["Heading2"]))
story.append(Spacer(1, 8))
story.append(Paragraph(
    f"{top_product['product']} was the highest-revenue product, generating "
    f"{top_product['revenue']:,.0f}. Total business profit was "
    f"{total_profit:,.0f}.",
    styles["BodyText"],
))

if (OUTPUT_DIR / "sales_chart.png").exists():
    story.extend([Spacer(1, 16), Paragraph("Revenue by Product", styles["Heading2"])])
    story.append(RLImage(str(OUTPUT_DIR / "sales_chart.png"), width=15 * cm, height=8.3 * cm))

doc.build(story)

print("PDF report created:", pdf_path)
print("File size:", round(pdf_path.stat().st_size / 1024, 2), "KB")

## Cell 26

In [ ]:
from google.colab import files

files.download(str(OUTPUT_DIR / "business_performance_report.pdf"))

## 11. Ask questions interactively

## Cell 27

In [ ]:
ask_business_agent("Which product generated the most revenue?")

## 12. Save and export the notebook

## Cell 28

In [ ]:
notebook_path = Path("/content/my_notebook.ipynb")

print("Notebook exists:", notebook_path.exists())
if notebook_path.exists():
    print("File size:", notebook_path.stat().st_size, "bytes")

## Cell 29

In [ ]:
from google.colab import _message

notebook_data = _message.blocking_request(
    "get_ipynb",
    request="",
    timeout_sec=30,
)["ipynb"]

notebook_path = Path("/content/my_notebook.ipynb")
with notebook_path.open("w", encoding="utf-8") as file:
    json.dump(notebook_data, file)

print("Saved:", notebook_path)
print("Size:", round(notebook_path.stat().st_size / 1024, 2), "KB")

## Cell 30

In [ ]:
from google.colab import files

files.download("/content/my_notebook.ipynb")

## Cell 31

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

drive_path = Path("/content/drive/MyDrive/ColabNotebooks/my_notebook.ipynb")
drive_path.parent.mkdir(parents=True, exist_ok=True)
drive_path.write_bytes(Path("/content/my_notebook.ipynb").read_bytes())

print("Notebook saved to:", drive_path)